# FairPlay AI · Clasificador de pares con datos reales de Steam

**Equipo FairPlay AI** · Ignacio Hidalgo, Sergio Rebolledo, Catalina Caris, Avelyn García · SIC 2026 IA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lina-cc/sic-ia-clone-radar/blob/main/FairPlay_AI_Clasificador_Pares.ipynb)

### Qué cambia respecto del notebook DBSCAN

| Antes (DBSCAN) | Ahora (este notebook) |
|---|---|
| 28 juegos inventados | Catálogo real de Steam (el parquet del equipo o el dataset de Kaggle) |
| Imágenes simuladas: los clones compartían un "centro" por construcción | Imágenes reales de Steam (header y capturas), comparadas con CLIP y con hash perceptual |
| Sin entrenamiento: solo `eps` ajustado sobre las mismas etiquetas | **Modelo entrenado y evaluado**: regresión logística sobre pares, validación cruzada por grupos |
| Pregunta: ¿qué juegos forman un grupo? | Pregunta: **¿este par de juegos merece revisión humana?** |

### El flujo

1. Cargar el catálogo y limpiar las descripciones.
2. Buscar **pares candidatos**: para cada juego, sus vecinos más cercanos por TF-IDF y por embeddings de texto.
3. Calcular **señales por par**: texto (TF-IDF y embeddings), imágenes (CLIP y hash perceptual), título, tags y fechas.
4. Armar etiquetas: **humanas** (la plantilla de registro y los casos guardados en la app) y **débiles** (automáticas, ver sección 5).
5. **Modelo A**: aprende a reconocer clones *sin mirar el texto*, usando las etiquetas débiles.
6. **Modelo B**: usa todas las señales y se entrena solo con etiquetas humanas.
7. Evaluar contra la línea base (solo TF-IDF), explicar pares, buscar candidatos y exportar pares nuevos para etiquetar.

> **Similitud no es plagio.** El modelo ordena pares para que una persona los revise. No acusa a nadie ni emite juicios legales.

**Cómo correrlo:** en Colab, activa la GPU (*Entorno de ejecución → Cambiar tipo → T4*), revisa la celda de configuración y ejecuta todo en orden.

## 0. Instalación y configuración

In [ ]:
# scikit-learn fijo en la misma versión que la app (Primera fase práctica/requirements.txt):
# un modelo guardado con una versión no siempre se puede usar con otra.
%pip install -q scikit-learn==1.9.1 sentence-transformers rapidfuzz kagglehub openpyxl imagehash

In [ ]:
# ── Configuración: lo único que normalmente hay que tocar ───────────────────
from pathlib import Path

FUENTE = "parquet"      # "parquet": el del equipo (data/juegos_texto_en.parquet)
                        # "kaggle":  dataset FronkonGames, se descarga solo con kagglehub
USAR_DRIVE = False      # True: lee y guarda todo en Google Drive (MyDrive/fairplay)

RUTA_PARQUET   = "data/juegos_texto_en.parquet"
RUTA_PLANTILLA = "data/Plantilla_registro_coincidencias.xlsx"  # opcional: casos revisados a mano
RUTA_CASOS_APP = "data/casos_guardados.csv"                    # opcional: casos marcados en la app Gradio
RUTA_LOTE      = "data/lote_etiquetado.xlsx"                   # opcional: lote de la sección 10 ya etiquetado

MAX_JUEGOS = None       # None = todos. Sin GPU, usa ~20000 (ojo: al recortar se pierden pares)
MIN_PALABRAS = 15       # descripciones más cortas se descartan
K_VECINOS = 10          # vecinos por juego para armar pares candidatos

UMBRAL_DEBIL_POS = 0.90         # TF-IDF >= esto: par "casi idéntico" (etiqueta débil positiva)
UMBRAL_DEBIL_EMB = 0.80         # ...y además embeddings >= esto (confirma que dicen lo mismo)
MIN_TERMINOS = 20               # términos TF-IDF mínimos por descripción para usarla en etiquetas débiles
RANGO_DEBIL_NEG = (0.20, 0.60)  # TF-IDF en este rango: mismo tema, texto distinto (débil negativa)
NEG_POR_POS = 3                 # negativos débiles por cada positivo
MAX_JUEGOS_IMG = 6000           # tope de juegos con imágenes por consulta (cuesta tiempo y red)
N_CAPTURAS = 3                  # capturas de pantalla por juego, además del header (0 = solo header)
CAPTURAS_API = False            # True: si el catálogo no trae capturas, pedirlas a la API de la tienda
                                # de Steam. Es lento (~1,5 s por juego) para respetar su límite.
N_LOTE = 100                    # pares del lote para etiquetar a mano (sección 10)
CABECERAS_APP = True            # sección 11: procesa la imagen de cabecera de TODOS los juegos para la app
                                # (~30 min la primera vez con GPU; después queda en caché)

MODELO_TEXTO  = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODELO_IMAGEN = "clip-ViT-B-32"
SEMILLA = 42

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/fairplay")
else:
    BASE = Path(".")

RUTA_PARQUET, RUTA_PLANTILLA, RUTA_CASOS_APP, RUTA_LOTE = (BASE / RUTA_PARQUET, BASE / RUTA_PLANTILLA,
                                                           BASE / RUTA_CASOS_APP, BASE / RUTA_LOTE)
SALIDA = BASE / "salidas"
CARPETA_IMG = SALIDA / "imagenes"
CARPETA_IMG.mkdir(parents=True, exist_ok=True)
print("Base:", BASE.resolve())

In [ ]:
import html
import io
import json
import re
import time
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations

import imagehash
import joblib
import sklearn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from PIL import Image
from rapidfuzz import fuzz
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(SEMILLA)
pd.set_option("display.max_colwidth", 80)

## 1. Catálogo de juegos

Se aceptan dos fuentes y ambas terminan con las mismas columnas:
`appid, name, text, developer, release_date, genres, tags, header_image, screenshots, n_resenas`.

`screenshots` es una lista de URLs de capturas de pantalla. El dataset de Kaggle la trae; el parquet del equipo probablemente no (en ese caso se puede activar `CAPTURAS_API`).
Las que no existan quedan vacías y el notebook sigue funcionando (por ejemplo, sin `developer` no se puede calcular `mismo_dev`).

In [ ]:
ALIAS = {
    "appid":        ["appid", "app_id", "steam_appid"],
    "name":         ["name", "titulo", "title"],
    "text":         ["text", "texto", "about_the_game", "about the game", "detailed_description",
                     "short_description", "descripcion"],
    "developer":    ["developer", "developers", "desarrollador"],
    "release_date": ["release_date", "release date", "fecha", "fecha_lanzamiento"],
    "genres":       ["genres", "generos", "géneros"],
    "tags":         ["tags", "etiquetas"],
    "header_image": ["header_image", "header image"],
    "screenshots":  ["screenshots", "capturas"],
    "n_resenas":    ["n_resenas", "reviews", "n_reviews"],
}


def a_lista_urls(v) -> list[str]:
    """Capturas en cualquier formato -> lista de URLs.
    Acepta: lista de strings, lista de dicts de la API ({'path_thumbnail': ...}),
    o un string con URLs separadas por comas (como en el CSV de Kaggle)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return []
    if isinstance(v, str):
        v = v.split(",")
    urls = []
    for x in v:
        if isinstance(x, dict):
            x = x.get("path_thumbnail") or x.get("path_full")
        if isinstance(x, str) and x.strip().startswith("http"):
            urls.append(x.strip())
    return urls


def limpiar(texto) -> str:
    """Misma limpieza que similitud.py: entidades HTML, etiquetas y espacios."""
    t = html.unescape(str(texto))
    t = re.sub(r"<[^>]+>", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def a_texto(v) -> str:
    """Listas, diccionarios (tags de Steam) o strings -> 'a, b, c'."""
    if isinstance(v, dict):
        return ", ".join(map(str, v.keys()))
    if isinstance(v, (list, tuple, np.ndarray)):
        return ", ".join(map(str, v))
    return "" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v)


def normalizar(crudo: pd.DataFrame) -> pd.DataFrame:
    minus = {c.lower().strip(): c for c in crudo.columns}
    df = pd.DataFrame(index=crudo.index)
    for destino, opciones in ALIAS.items():
        origen = next((minus[o] for o in opciones if o in minus), None)
        df[destino] = crudo[origen] if origen is not None else None
    for c in ["name", "developer", "genres", "tags", "header_image"]:
        df[c] = df[c].map(a_texto)
    df["screenshots"] = df["screenshots"].map(a_lista_urls)
    df["text"] = df["text"].map(limpiar)
    df["appid"] = pd.to_numeric(df["appid"], errors="coerce").astype("Int64")
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce", format="mixed")
    df["n_resenas"] = pd.to_numeric(df["n_resenas"], errors="coerce")
    return df


def cargar_kaggle() -> pd.DataFrame:
    """Dataset 'fronkongames/steam-games-dataset' (games.json). Si cambian las claves, ajusta aquí.

    Leer games.json ocupa varios GB de RAM, así que la primera vez se guarda una versión reducida
    en salidas/kaggle_catalogo.parquet y las siguientes corridas leen esa (segundos, poca RAM)."""
    cache = SALIDA / "kaggle_catalogo.parquet"
    if cache.exists():
        return pd.read_parquet(cache)
    import kagglehub
    carpeta = Path(kagglehub.dataset_download("fronkongames/steam-games-dataset"))
    with open(next(carpeta.rglob("games.json")), encoding="utf-8") as f:
        datos = json.load(f)
    filas = []
    for appid, g in datos.items():
        filas.append({
            "appid": appid, "name": g.get("name"),
            "text": g.get("about_the_game") or g.get("detailed_description") or g.get("short_description"),
            "developer": g.get("developers"), "release_date": g.get("release_date"),
            "genres": g.get("genres"), "tags": g.get("tags"), "header_image": g.get("header_image"),
            "screenshots": a_lista_urls(g.get("screenshots"))[:max(N_CAPTURAS, 1)],
            "n_resenas": (g.get("positive") or 0) + (g.get("negative") or 0),
        })
    del datos
    tabla = pd.DataFrame(filas)
    for c in ["developer", "genres", "tags"]:          # listas/diccionarios -> texto (parquet-friendly)
        tabla[c] = tabla[c].map(a_texto)
    tabla.to_parquet(cache)
    return tabla


crudo = pd.read_parquet(RUTA_PARQUET) if FUENTE == "parquet" else cargar_kaggle()
df = normalizar(crudo)
del crudo

n0 = len(df)
df = df[df["text"].str.split().str.len() >= MIN_PALABRAS]
df = df[df["appid"].isna() | ~df["appid"].duplicated()]
df = df.reset_index(drop=True)
print(f"Juegos: {n0:,} cargados -> {len(df):,} con descripción de al menos {MIN_PALABRAS} palabras")
con_datos = [c for c in df.columns if c != "screenshots"
             and df[c].notna().any() and df[c].astype(str).str.len().gt(0).any()]
print("Columnas con datos:", con_datos)
print(f"Juegos con capturas en el catálogo: {int(df['screenshots'].str.len().gt(0).sum()):,}")

### Etiquetas humanas (se cargan antes de recortar el catálogo)

* **Plantilla de registro** (`Plantilla_registro_coincidencias.xlsx`): cada `ID_caso` agrupa juegos; todos sus pares se etiquetan según `Etiqueta_humana`.
  *Coincidencia relevante* → 1 · *Caso legítimo* y *Descartado* → 0 · *No concluyente* → se ignora · **vacío → 1 provisional** (los casos cargados hasta ahora son descripciones idénticas de estudios distintos; confirmen la etiqueta cuando los revisen).
* **Casos de la app Gradio** (`casos_guardados.csv`): *Posible copia* → 1 · *Similar* y *No similar* → 0. *Similar* es justamente el negativo difícil: se parecen, pero no es copia.

In [ ]:
MAPA_PLANTILLA = {"coincidencia relevante": 1, "caso legítimo": 0, "caso legitimo": 0, "descartado": 0}
MAPA_APP = {"posible copia": 1, "similar": 0, "no similar": 0}


def leer_plantilla(ruta) -> list[dict]:
    if not Path(ruta).exists():
        print("Sin plantilla:", ruta)
        return []
    resumen = pd.read_excel(ruta, sheet_name="Resumen de casos")
    juegos = pd.read_excel(ruta, sheet_name="Juegos")
    etiqueta = dict(zip(resumen["ID_caso"], resumen["Etiqueta_humana"]))
    pares = []
    for caso, grupo in juegos.dropna(subset=["appid"]).groupby("ID_caso"):
        e = str(etiqueta.get(caso, "") if pd.notna(etiqueta.get(caso, "")) else "").strip().lower()
        if e == "no concluyente":
            continue
        y, prov = (MAPA_PLANTILLA[e], False) if e in MAPA_PLANTILLA else (1, True)
        for a, b in combinations(sorted(int(x) for x in grupo["appid"]), 2):
            pares.append({"appid_a": a, "appid_b": b, "y": y, "fuente": "plantilla",
                          "provisional": prov, "caso": caso})
    return pares


def leer_casos_app(ruta) -> list[dict]:
    if not Path(ruta).exists():
        print("Sin casos de la app:", ruta)
        return []
    c = pd.read_csv(ruta, encoding="utf-8-sig")
    c = c[c["etiqueta"].str.lower().isin(MAPA_APP)]
    return [{"nombre_a": r.consulta, "nombre_b": r.juego_comparado, "y": MAPA_APP[r.etiqueta.lower()],
             "fuente": f"app ({r.anotador})", "provisional": False, "caso": f"app-{k}"}
            for k, r in enumerate(c.itertuples())]


def leer_lote(ruta) -> list[dict]:
    """Lote de la sección 10 (hoja "Pares") con la columna Etiqueta_humana llena."""
    if not Path(ruta).exists():
        print("Sin lote etiquetado:", ruta)
        return []
    lote = pd.read_excel(ruta, sheet_name="Pares")
    lote["e"] = lote["Etiqueta_humana"].fillna("").astype(str).str.strip().str.lower()
    lote = lote[lote["e"].isin(MAPA_PLANTILLA)].dropna(subset=["appid_a", "appid_b"])
    return [{"appid_a": int(r.appid_a), "appid_b": int(r.appid_b), "y": MAPA_PLANTILLA[r.e],
             "fuente": f"lote ({r.Responsable})" if pd.notna(r.Responsable) else "lote",
             "provisional": False, "caso": r.ID_caso} for r in lote.itertuples()]


humanos_crudos = leer_plantilla(RUTA_PLANTILLA) + leer_casos_app(RUTA_CASOS_APP) + leer_lote(RUTA_LOTE)
print(f"Pares humanos leídos: {len(humanos_crudos)}")

# Recorte opcional del catálogo, sin perder los juegos etiquetados a mano
if MAX_JUEGOS and len(df) > MAX_JUEGOS:
    ids_h = {p[k] for p in humanos_crudos for k in ("appid_a", "appid_b") if k in p}
    nombres_h = {str(p[k]).lower() for p in humanos_crudos for k in ("nombre_a", "nombre_b") if k in p}
    forzar = df["appid"].isin(ids_h) | df["name"].str.lower().isin(nombres_h)
    resto = df[~forzar].sample(n=max(MAX_JUEGOS - int(forzar.sum()), 0), random_state=SEMILLA)
    df = pd.concat([df[forzar], resto]).reset_index(drop=True)
    print(f"Catálogo recortado a {len(df):,} juegos ({int(forzar.sum())} forzados por estar etiquetados)")

## 2. Pares candidatos: vecinos por TF-IDF y por embeddings

Comparar todos contra todos es imposible (100 mil juegos ≈ 5 mil millones de pares). Por eso, para cada juego se buscan sus `K_VECINOS` más cercanos con dos métodos y se unen:

* **TF-IDF** (la misma configuración que la app): encuentra textos con las mismas palabras.
* **Embeddings multilingües** (`paraphrase-multilingual-MiniLM-L12-v2`, Hugging Face): encuentra textos con el mismo *significado* aunque cambien las palabras (los casos M1 y M2 de la plantilla).

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2,
                        max_features=50000, sublinear_tf=True)
X = tfidf.fit_transform(df["text"])
VOCAB = tfidf.get_feature_names_out()
print("TF-IDF:", X.shape)
# Términos distintos que el TF-IDF reconoce en cada descripción. Las descripciones en otros
# idiomas (chino, ruso...) quedan casi vacías: con 1 o 2 términos, dos textos distintos
# pueden dar tfidf_cos = 1. Se usa para no crear etiquetas débiles falsas (sección 5).
N_TERMINOS = np.diff(X.tocsr().indptr)
print(f"Descripciones con menos de {MIN_TERMINOS} términos TF-IDF: {(N_TERMINOS < MIN_TERMINOS).mean():.1%}")

import torch
from sentence_transformers import SentenceTransformer

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DISPOSITIVO)
cache_E = SALIDA / f"emb_texto_{len(df)}.npy"
if cache_E.exists():
    E = np.load(cache_E)
else:
    modelo_txt = SentenceTransformer(MODELO_TEXTO, device=DISPOSITIVO)
    E = modelo_txt.encode(df["text"].str.slice(0, 2000).tolist(), batch_size=128,
                          normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    np.save(cache_E, E)
print("Embeddings de texto:", E.shape)

In [ ]:
def vecinos(M, k, lote=2048, dispositivo=None):
    """Índices de los k+1 juegos más parecidos (coseno) a cada juego, incluido él mismo.

    Las filas de M ya tienen norma 1 (TF-IDF y embeddings vienen normalizados), así que
    coseno = producto punto. Con GPU se calcula por lotes con torch: segundos o pocos minutos
    para todo Steam. Sin GPU se usa scikit-learn, que con más de ~50 mil juegos tarda mucho."""
    from scipy import sparse
    dispositivo = dispositivo or DISPOSITIVO
    n, k = M.shape[0], min(k + 1, M.shape[0])
    if dispositivo == "cpu":
        nn = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="brute", n_jobs=-1).fit(M)
        return nn.kneighbors(M, return_distance=False)
    salida = np.empty((n, k), dtype=np.int64)
    with torch.no_grad():
        if sparse.issparse(M):
            M = M.tocsr().astype(np.float32)
            base = torch.sparse_csr_tensor(torch.from_numpy(M.indptr.astype(np.int64)),
                                           torch.from_numpy(M.indices.astype(np.int64)),
                                           torch.from_numpy(M.data), size=M.shape).to(dispositivo)
            for s in range(0, n, lote):
                bloque = torch.from_numpy(M[s:s + lote].toarray()).to(dispositivo)
                sim = (base @ bloque.T).T                      # (lote x n)
                salida[s:s + lote] = sim.topk(k, dim=1).indices.cpu().numpy()
        else:
            base = torch.from_numpy(np.ascontiguousarray(M, dtype=np.float32)).to(dispositivo)
            for s in range(0, n, lote):
                sim = base[s:s + lote] @ base.T
                salida[s:s + lote] = sim.topk(k, dim=1).indices.cpu().numpy()
    return salida


def a_pares(ind) -> pd.DataFrame:
    i = np.repeat(np.arange(ind.shape[0]), ind.shape[1])
    j = ind.ravel()
    m = i != j
    return pd.DataFrame({"i": np.minimum(i[m], j[m]), "j": np.maximum(i[m], j[m])})


cache_vec = SALIDA / f"vecinos_{len(df)}_{K_VECINOS}.npz"   # se recalcula si cambia el catálogo
if cache_vec.exists():
    _v = np.load(cache_vec)
    IND_TFIDF, IND_EMB = _v["tfidf"], _v["emb"]
else:
    t0 = time.time()
    IND_TFIDF = vecinos(X, K_VECINOS)
    IND_EMB = vecinos(E, K_VECINOS)
    np.savez(cache_vec, tfidf=IND_TFIDF, emb=IND_EMB)
    print(f"Vecinos calculados en {time.time() - t0:.0f} s ({DISPOSITIVO})")
pares = pd.concat([a_pares(IND_TFIDF), a_pares(IND_EMB)]).drop_duplicates().reset_index(drop=True)
print(f"Pares candidatos: {len(pares):,}")

## 3. Señales por par

| Señal | Qué mide | Rango |
|---|---|---|
| `tfidf_cos` | mismas palabras en la descripción | 0–1 |
| `emb_cos` | mismo significado en la descripción | ~0–1 |
| `img_cos` | parecido visual (CLIP): el par de imágenes más parecido entre header y capturas | ~0–1, vacío si no hay imagen |
| `phash_sim` | imágenes casi idénticas (hash perceptual): detecta assets reutilizados | 0–1, vacío si no hay imagen |
| `titulo_sim` | parecido del nombre (`token_set_ratio`) | 0–1 |
| `tags_jaccard` | géneros y tags en común | 0–1 |
| `log_dias` | distancia entre fechas de lanzamiento, log(1 + días) | ≥ 0 |
| `mismo_dev` | mismo desarrollador (una serie propia no es clon) | 0/1 |

Las señales que faltan quedan vacías (`NaN`). El modelo las rellena con la mediana y agrega una columna que avisa que faltaban.

In [ ]:
TAGS = [frozenset(t.strip().lower() for t in f"{g}, {t}".split(",") if t.strip())
        for g, t in zip(df["genres"], df["tags"])]
DEV = df["developer"].fillna("").str.lower().str.strip().to_numpy()
FECHA = df["release_date"].to_numpy()
NOMBRE = df["name"].fillna("").to_numpy()


def por_lotes(f, a, b, lote=200_000) -> np.ndarray:
    """Aplica f(a, b) por partes: con millones de pares, copiar todas las filas de una vez
    llenaría la RAM de Colab."""
    if len(a) == 0:
        return np.zeros(0)
    return np.concatenate([f(a[s:s + lote], b[s:s + lote]) for s in range(0, len(a), lote)])


def cos_sparse(M, a, b):
    return por_lotes(lambda x, y: np.asarray(M[x].multiply(M[y]).sum(axis=1)).ravel(), a, b)


def cos_denso(M, a, b):
    return por_lotes(lambda x, y: np.einsum("ij,ij->i", M[x], M[y]), a, b)


def senales_texto(p: pd.DataFrame) -> pd.DataFrame:
    """Todas las señales menos la de imagen (que necesita descargar)."""
    a, b = p["i"].to_numpy(), p["j"].to_numpy()
    out = p[["i", "j"]].copy()
    out["tfidf_cos"] = cos_sparse(X, a, b)
    out["emb_cos"] = cos_denso(E, a, b)
    out["titulo_sim"] = [fuzz.token_set_ratio(NOMBRE[x], NOMBRE[y]) / 100 for x, y in zip(a, b)]
    out["tags_jaccard"] = [len(TAGS[x] & TAGS[y]) / len(TAGS[x] | TAGS[y]) if TAGS[x] | TAGS[y] else np.nan
                           for x, y in zip(a, b)]
    dias = np.abs((pd.to_datetime(FECHA[a]) - pd.to_datetime(FECHA[b])).days.to_numpy(dtype=float))
    out["log_dias"] = np.log1p(dias)
    conocido = (DEV[a] != "") & (DEV[b] != "")
    out["mismo_dev"] = np.where(conocido, (DEV[a] == DEV[b]).astype(float), np.nan)
    return out


pares = senales_texto(pares)
pares.describe().T.round(3)

## 4. Imágenes reales: header, capturas, CLIP y hash perceptual

Para cada juego se usan hasta `1 + N_CAPTURAS` imágenes públicas de Steam: el **header** (la imagen de cabecera de la tienda) y las primeras **capturas de pantalla**. Las capturas vienen del catálogo (Kaggle las trae) o, si se activa `CAPTURAS_API`, de la API pública de la tienda (`store.steampowered.com/api/appdetails`), guardando cada respuesta en `salidas/api/` para no repetir consultas.

Cada imagen se resume de dos formas, porque miden cosas distintas:

| Técnica | Qué detecta | Ejemplo |
|---|---|---|
| **CLIP** (`clip-ViT-B-32`, preentrenado) | imágenes que *se ven parecidas*: mismo estilo, misma escena, mismos elementos | dos juegos de aviones con cielo y nubes |
| **Hash perceptual** (`imagehash.phash`) | imágenes *casi idénticas*, aunque cambie el tamaño o la compresión | la misma captura o el mismo asset reutilizado |

Para un par de juegos se comparan **todas sus imágenes contra todas** y se guarda el mejor resultado:
* `img_cos` = la mayor similitud CLIP entre cualquier imagen de A y cualquier imagen de B.
* `phash_sim` = 1 − (menor distancia de Hamming entre hashes) / 64. Vale 1 si hay una imagen idéntica.

CLIP es *transfer learning*: no entrenamos la red de visión, usamos lo que ya aprendió. El hash perceptual no usa aprendizaje: es una huella de 64 bits de la imagen reducida.

Solo se descargan las imágenes de los juegos que se necesitan (pares etiquetados, búsquedas y exportación), con un tope de `MAX_JUEGOS_IMG` juegos.

In [ ]:
URL_HEADER = "https://cdn.akamai.steamstatic.com/steam/apps/{appid}/header.jpg"
URL_API = "https://store.steampowered.com/api/appdetails?appids={appid}"
CABECERAS = {"User-Agent": "FairPlayAI-capstone/0.1 (proyecto educativo)"}
PAUSA_API = 1.5                                 # segundos entre consultas a la API de la tienda
CARPETA_API = SALIDA / "api"
CARPETA_API.mkdir(parents=True, exist_ok=True)

# IMGS[k] = {"emb": matriz (n_imagenes x 512) CLIP, "hash": matriz (n_imagenes x 64) de bits,
#            "rutas": lista de archivos}. k es la posición del juego en df.
IMGS: dict[int, dict] = {}
cache_img = SALIDA / "imagenes_v2.joblib"       # se guarda por appid para sobrevivir a recortes del catálogo
if cache_img.exists():
    _pos = {int(a): k for k, a in df["appid"].dropna().items()}
    IMGS = {_pos[a]: v for a, v in joblib.load(cache_img).items() if a in _pos}
    print(f"Juegos con imágenes en caché: {len(IMGS)}")
_modelo_img = None


def capturas_api(appid: int) -> list[str]:
    """Capturas desde la API de la tienda. Respuesta cacheada en disco; pausa para no saturarla."""
    archivo = CARPETA_API / f"{appid}.json"
    if not archivo.exists():
        time.sleep(PAUSA_API)
        try:
            r = requests.get(URL_API.format(appid=appid), headers=CABECERAS, timeout=15)
            archivo.write_text(r.text, encoding="utf-8")
        except Exception:
            return []
    try:
        datos = json.loads(archivo.read_text(encoding="utf-8")).get(str(appid), {})
        return a_lista_urls(datos.get("data", {}).get("screenshots")) if datos.get("success") else []
    except Exception:
        return []


def urls_de(k: int) -> list[str]:
    """Header + hasta N_CAPTURAS capturas del juego k (posición 0 = header)."""
    a = df.at[k, "appid"]
    header = df.at[k, "header_image"]
    if not (isinstance(header, str) and header.startswith("http")):
        header = URL_HEADER.format(appid=int(a)) if pd.notna(a) else None
    capturas = list(df.at[k, "screenshots"])
    if not capturas and CAPTURAS_API and pd.notna(a) and N_CAPTURAS > 0:
        capturas = capturas_api(int(a))
    return ([header] if header else []) + capturas[:N_CAPTURAS]


def ruta_img(k: int, n: int = 0) -> Path:
    a = df.at[k, "appid"]
    return CARPETA_IMG / (f"{int(a)}_{n}.jpg" if pd.notna(a) else f"idx{k}_{n}.jpg")


def descargar(tarea):
    """tarea = (k, n, url). Devuelve la ruta local o None si no se pudo."""
    k, n, url = tarea
    ruta = ruta_img(k, n)
    if ruta.exists():
        return ruta
    try:
        r = requests.get(url, headers=CABECERAS, timeout=15)
        r.raise_for_status()
        Image.open(io.BytesIO(r.content)).convert("RGB").save(ruta, "JPEG")
        return ruta
    except Exception:
        return None


def hash_bits(img: Image.Image) -> np.ndarray:
    """Hash perceptual de 64 bits como vector booleano."""
    return imagehash.phash(img).hash.flatten()


def asegurar_imagenes(indices) -> None:
    """Descarga y procesa (CLIP + hash) las imágenes de los juegos que falten."""
    global _modelo_img
    pedidos = list(dict.fromkeys(int(k) for k in indices))
    faltan = [k for k in pedidos if k not in IMGS]
    # El tope cuenta solo los juegos de esta consulta (no la caché acumulada de corridas anteriores):
    # si contara la caché, los pares nuevos se quedarían sin imágenes.
    cupo = max(MAX_JUEGOS_IMG - (len(pedidos) - len(faltan)), 0)
    if len(faltan) > cupo:
        print(f"Aviso: se piden imágenes de {len(pedidos)} juegos y el tope es {MAX_JUEGOS_IMG} (MAX_JUEGOS_IMG).")
        faltan = faltan[:cupo]
    if not faltan:
        return
    # 1) URLs (secuencial: puede consultar la API de la tienda con pausas)
    tareas = [(k, n, u) for k in faltan for n, u in enumerate(urls_de(k))]
    # 2) Descargas en paralelo
    with ThreadPoolExecutor(max_workers=16) as ex:
        rutas = list(ex.map(descargar, tareas))
    por_juego: dict[int, list[Path]] = {}
    for (k, _, _), r in zip(tareas, rutas):
        if r is not None:
            por_juego.setdefault(k, []).append(r)
    print(f"Imágenes: {sum(map(len, por_juego.values()))} descargadas de {len(tareas)} pedidas "
          f"({len(por_juego)} de {len(faltan)} juegos con al menos una)")
    if not por_juego:
        return
    # 3) CLIP + hash perceptual
    if _modelo_img is None:
        _modelo_img = SentenceTransformer(MODELO_IMAGEN, device=DISPOSITIVO)
    lista = [(k, r) for k, rs in por_juego.items() for r in rs]
    for desde in range(0, len(lista), 64):
        lote = lista[desde:desde + 64]
        imgs = [Image.open(r).convert("RGB") for _, r in lote]
        V = _modelo_img.encode(imgs, batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
        for (k, r), v, im in zip(lote, V, imgs):
            d = IMGS.setdefault(k, {"emb": [], "hash": [], "rutas": []})
            d["emb"].append(v)
            d["hash"].append(hash_bits(im))
            d["rutas"].append(str(r))
    for k in por_juego:
        IMGS[k]["emb"] = np.vstack(IMGS[k]["emb"])
        IMGS[k]["hash"] = np.vstack(IMGS[k]["hash"])
    joblib.dump({int(df.at[k, "appid"]): v for k, v in IMGS.items() if pd.notna(df.at[k, "appid"])},
                cache_img)


def comparar_imagenes(a: int, b: int):
    """(img_cos, phash_sim, (fila_a, fila_b) del par CLIP más parecido) o NaN si falta alguna."""
    if a not in IMGS or b not in IMGS:
        return np.nan, np.nan, None
    S = IMGS[a]["emb"] @ IMGS[b]["emb"].T
    H = (IMGS[a]["hash"][:, None, :] != IMGS[b]["hash"][None, :, :]).sum(axis=2)
    mejor = np.unravel_index(S.argmax(), S.shape)
    return float(S.max()), 1 - float(H.min()) / 64, mejor


def senal_imagen(p: pd.DataFrame) -> pd.DataFrame:
    asegurar_imagenes(pd.concat([p["i"], p["j"]]).tolist())
    p = p.copy()
    res = [comparar_imagenes(a, b) for a, b in zip(p["i"], p["j"])]
    p["img_cos"] = [r[0] for r in res]
    p["phash_sim"] = [r[1] for r in res]
    return p

## 5. Etiquetas

### Humanas
Las leídas en la sección 1, ahora traducidas a posiciones del catálogo.

### Débiles (automáticas)
Con pocas etiquetas humanas no se puede entrenar, así que generamos etiquetas aproximadas:

* **Positivo débil:** descripción casi idéntica (`tfidf_cos ≥ 0.90`) **y** con el mismo significado (`emb_cos ≥ 0.80`), entre **estudios distintos**. Es el patrón de los casos C-001 a C-005 de la plantilla.
* **Negativo débil:** pares del mismo tema pero con texto claramente distinto (`0.20 ≤ tfidf_cos ≤ 0.60`).
* **Solo descripciones informativas:** ambas con al menos `MIN_TERMINOS` términos TF-IDF.

**Por qué los dos últimos filtros:** en la primera corrida con todo Steam, la mitad de los positivos débiles eran falsos. Eran juegos con la descripción en chino u otro idioma: el TF-IDF (pensado para inglés) los deja con 1 o 2 términos, y dos textos distintos terminaban con `tfidf_cos = 1`. Los embeddings multilingües sí entienden esos textos y mostraban que no se parecían (`emb_cos` ≈ 0,3). Exigir ambas cosas deja solo copias de verdad.

**Trampa a evitar:** como estas etiquetas se definen *con* el texto, un modelo que viera `tfidf_cos` acertaría siempre sin aprender nada. Por eso el **Modelo A** no ve ninguna señal de texto: aprende qué *otras* señales (imágenes, título, tags, fechas) acompañan a un clon. Así puede detectar clones con la descripción reescrita, que es justo lo que TF-IDF no ve.

In [ ]:
POS_DE_APPID = {int(a): k for k, a in df["appid"].dropna().items()}
POS_DE_NOMBRE = {}
for k in df.sort_values("n_resenas", ascending=True, na_position="first").index:
    POS_DE_NOMBRE[str(df.at[k, "name"]).strip().lower()] = k   # homónimos: gana el más reseñado

filas_h, sin_mapear = [], 0
for p in humanos_crudos:
    if "appid_a" in p:
        a, b = POS_DE_APPID.get(p["appid_a"]), POS_DE_APPID.get(p["appid_b"])
    else:
        a = POS_DE_NOMBRE.get(str(p["nombre_a"]).strip().lower())
        b = POS_DE_NOMBRE.get(str(p["nombre_b"]).strip().lower())
    if a is None or b is None or a == b:
        sin_mapear += 1
        continue
    filas_h.append({"i": min(a, b), "j": max(a, b), "y": p["y"], "fuente": p["fuente"],
                    "provisional": p["provisional"], "caso": p["caso"]})
humanos = pd.DataFrame(filas_h, columns=["i", "j", "y", "fuente", "provisional", "caso"])
humanos = humanos.drop_duplicates(["i", "j"], keep="last").reset_index(drop=True)
print(f"Pares humanos en el catálogo: {len(humanos)} (positivos {int(humanos.y.sum())}, "
      f"negativos {int((humanos.y == 0).sum())}); sin encontrar en el catálogo: {sin_mapear}")

# Etiquetas débiles
ya = set(zip(humanos["i"], humanos["j"]))
libre = pares[[(i, j) not in ya for i, j in zip(pares["i"], pares["j"])]]
informativo = ((N_TERMINOS[libre["i"].to_numpy()] >= MIN_TERMINOS)
               & (N_TERMINOS[libre["j"].to_numpy()] >= MIN_TERMINOS))
pos = libre[informativo & (libre["tfidf_cos"] >= UMBRAL_DEBIL_POS).to_numpy()
            & (libre["emb_cos"] >= UMBRAL_DEBIL_EMB).to_numpy() & (libre["mismo_dev"] != 1).to_numpy()]
neg = libre[informativo & libre["tfidf_cos"].between(*RANGO_DEBIL_NEG).to_numpy()]
print(f"Pares descartados por descripción casi vacía (pocos términos TF-IDF): {(~informativo).sum():,}")
n_pos = min(len(pos), MAX_JUEGOS_IMG // (2 * (1 + NEG_POR_POS)))   # cada par usa hasta 2 juegos con imágenes
pos = pos.sample(n=n_pos, random_state=SEMILLA)
neg = neg.sample(n=min(len(neg), n_pos * NEG_POR_POS), random_state=SEMILLA)
debiles = pd.concat([pos.assign(y=1), neg.assign(y=0)]).reset_index(drop=True)
print(f"Pares débiles: {len(debiles)} (positivos {len(pos)}, negativos {len(neg)})")

In [ ]:
debiles = senal_imagen(debiles)
# El Modelo A se entrena solo con pares que tienen imagen. Si no, puede aprender una regla falsa:
# "sin imagen = no es copia" (pasa si las imágenes faltan más en una clase que en la otra).
sin_img = debiles["img_cos"].isna()
print("Pares débiles sin imagen, descartados:", int(sin_img.sum()),
      dict(debiles.loc[sin_img, "y"].value_counts().sort_index()))
debiles = debiles[~sin_img].reset_index(drop=True)
if len(humanos):
    humanos = senal_imagen(senales_texto(humanos[["i", "j"]]).join(humanos[["y", "fuente", "provisional", "caso"]]))
print(f"Pares débiles para el Modelo A: {len(debiles)} (positivos {int(debiles.y.sum())}, "
      f"negativos {int((debiles.y == 0).sum())})")
debiles.groupby("y")[["tfidf_cos", "emb_cos", "img_cos", "phash_sim", "titulo_sim", "tags_jaccard",
                     "log_dias"]].mean().round(3)

## 6. Modelo A: reconoce clones sin mirar el texto

* **Entrada:** `img_cos`, `phash_sim`, `titulo_sim`, `tags_jaccard`, `log_dias`.
* **Modelo:** regresión logística (sus coeficientes dicen cuánto pesa cada señal) comparada con *gradient boosting*.
* **Validación:** `GroupKFold`. Los pares que comparten juegos de una misma familia quedan en el mismo grupo, así un clon no se evalúa con su familia vista en el entrenamiento.
* **Métricas:** *PR-AUC* (precisión promedio, la adecuada cuando los positivos son pocos) y ROC-AUC.

In [ ]:
FEAT_A = ["img_cos", "phash_sim", "titulo_sim", "tags_jaccard", "log_dias"]   # sin señales de texto
FEAT_B = ["tfidf_cos", "emb_cos"] + FEAT_A + ["mismo_dev"]


def logistica():
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True),
                         StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))


def boosting():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.1, max_iter=200,
                                          class_weight="balanced", random_state=SEMILLA)


def grupos_por_familia(p: pd.DataFrame) -> np.ndarray:
    """Union-find: juegos unidos por pares positivos forman una familia."""
    padre = {}
    def raiz(x):
        padre.setdefault(x, x)
        while padre[x] != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x
    for i, j in p.loc[p["y"] == 1, ["i", "j"]].itertuples(index=False):
        padre[raiz(i)] = raiz(j)
    return np.array([raiz(i) for i in p["i"]])


def precision_en_k(y, s, k):
    k = min(k, len(y))
    return float(np.asarray(y)[np.argsort(-np.asarray(s))[:k]].mean()) if k else float("nan")


def reporte(nombre, y, s):
    return {"modelo": nombre, "PR-AUC": average_precision_score(y, s), "ROC-AUC": roc_auc_score(y, s),
            "precisión@20": precision_en_k(y, s, 20), "positivos": int(np.sum(y)), "pares": len(y)}


yA = debiles["y"].to_numpy()
if len(np.unique(yA)) < 2:
    raise RuntimeError("Las etiquetas débiles tienen una sola clase: revisa UMBRAL_DEBIL_POS / RANGO_DEBIL_NEG.")
gkf = GroupKFold(n_splits=min(5, len(np.unique(grupos_por_familia(debiles)))))
filas = []
for nombre, fabrica in [("A · logística", logistica), ("A · boosting", boosting)]:
    s = cross_val_predict(fabrica(), debiles[FEAT_A], yA, groups=grupos_por_familia(debiles),
                          cv=gkf, method="predict_proba")[:, 1]
    filas.append(reporte(nombre, yA, s))
filas.append(reporte("Azar (referencia)", yA, rng.random(len(yA))))
print("Validación cruzada sobre etiquetas débiles (pares nunca vistos en el entrenamiento):")
pd.DataFrame(filas).round(3)

In [ ]:
modelo_A = logistica().fit(debiles[FEAT_A], yA)
nombres = modelo_A[:-1].get_feature_names_out(FEAT_A)
coef = pd.Series(modelo_A[-1].coef_[0], index=nombres).sort_values()
ax = coef.plot.barh(figsize=(6, 3), color=np.where(coef > 0, "#2a9d8f", "#e76f51"))
ax.set_title("Modelo A: peso de cada señal (positivo = más probable que sea clon)")
ax.axvline(0, color="grey", lw=0.8)
plt.tight_layout()
plt.show()

## 7. Evaluación con etiquetas humanas y Modelo B

Esta es la evaluación que vale para el informe: **pares revisados por personas**, que el modelo nunca usó para definir sus etiquetas débiles.

Se compara:
* **Línea base:** solo `tfidf_cos` (lo que hace hoy la app).
* **Embeddings:** solo `emb_cos`.
* **Modelo A:** entrenado con etiquetas débiles, sin texto.
* **Modelo B:** todas las señales, entrenado y evaluado **solo con etiquetas humanas** mediante validación cruzada. Necesita al menos ~10 pares de cada clase.

Si faltan etiquetas, la celda lo dice. La sección 10 genera pares listos para etiquetar.

In [ ]:
modelo_B = None
n_pos_h, n_neg_h = int(humanos["y"].sum()), int((humanos["y"] == 0).sum()) if len(humanos) else 0
if len(humanos) == 0 or min(n_pos_h, n_neg_h) == 0:
    print(f"Todavía no se puede evaluar con etiquetas humanas: hay {n_pos_h} positivos y {n_neg_h} negativos.\n"
          "Etiqueten pares en la app ('Similar', 'No similar', 'Posible copia') o con el CSV de la sección 10,\n"
          "y vuelvan a correr desde la sección 1.")
else:
    yH = humanos["y"].to_numpy()
    filas = [reporte("Línea base: TF-IDF", yH, humanos["tfidf_cos"]),
             reporte("Embeddings de texto", yH, humanos["emb_cos"]),
             reporte("Modelo A (sin texto)", yH, modelo_A.predict_proba(humanos[FEAT_A])[:, 1])]
    if min(n_pos_h, n_neg_h) >= 10:
        cv = StratifiedKFold(n_splits=min(5, n_neg_h, n_pos_h), shuffle=True, random_state=SEMILLA)
        sB = cross_val_predict(logistica(), humanos[FEAT_B], yH, cv=cv, method="predict_proba")[:, 1]
        filas.append(reporte("Modelo B (todas, humano)", yH, sB))
        modelo_B = logistica().fit(humanos[FEAT_B], yH)
    else:
        print(f"Modelo B necesita al menos 10 pares de cada clase (hay {n_pos_h} y {n_neg_h}).")
    if humanos["provisional"].any():
        print(f"Ojo: {int(humanos['provisional'].sum())} pares tienen etiqueta provisional (plantilla sin revisar).")
    display(pd.DataFrame(filas).round(3))

## 8. Explicar un par

Para la revisión humana no basta un número. Esta función muestra:
1. Las señales del par.
2. Las palabras que más aportan a la similitud TF-IDF (lo mismo que muestra la app).
3. Cuánto empuja cada señal la probabilidad del Modelo A.
4. Los dos headers lado a lado.

In [ ]:
def terminos_compartidos(a, b, n=10):
    prod = X[a].multiply(X[b]).toarray().ravel()
    orden = prod.argsort()[::-1][:n]
    return [str(VOCAB[t]) for t in orden if prod[t] > 0]


def aportes_A(fila):
    """Contribución de cada señal al logit del Modelo A (valor estandarizado × coeficiente)."""
    z = modelo_A[:-1].transform(fila[FEAT_A].astype(float).to_frame().T)[0]
    return pd.Series(z * modelo_A[-1].coef_[0], index=nombres).sort_values(key=abs, ascending=False)


def explicar(a, b, mostrar_imagenes=True):
    a, b = (a, b) if a < b else (b, a)
    fila = senal_imagen(senales_texto(pd.DataFrame({"i": [a], "j": [b]}))).iloc[0]
    print(f"A: {df.at[a, 'name']}  ·  {df.at[a, 'developer']}  ·  {str(df.at[a, 'release_date'])[:10]}")
    print(f"B: {df.at[b, 'name']}  ·  {df.at[b, 'developer']}  ·  {str(df.at[b, 'release_date'])[:10]}\n")
    print(fila[["tfidf_cos", "emb_cos", "img_cos", "phash_sim", "titulo_sim", "tags_jaccard", "log_dias",
                "mismo_dev"]].astype(float).round(3).to_string())
    print("\nPalabras que más aportan:", ", ".join(terminos_compartidos(a, b)) or "(ninguna)")
    print(f"\nModelo A: probabilidad de revisar = {modelo_A.predict_proba(fila[FEAT_A].astype(float).to_frame().T)[0, 1]:.2f}")
    print(aportes_A(fila).round(2).to_string())
    if modelo_B is not None:
        print(f"Modelo B: probabilidad de revisar = {modelo_B.predict_proba(fila[FEAT_B].astype(float).to_frame().T)[0, 1]:.2f}")
    _, _, mejor = comparar_imagenes(a, b)
    if mostrar_imagenes and mejor is not None:
        # Se muestra el par de imágenes (header o captura) que CLIP encontró más parecido
        fig, ejes = plt.subplots(1, 2, figsize=(10, 2.8))
        for ax, k, fila_img in zip(ejes, (a, b), mejor):
            ruta = Path(IMGS[k]["rutas"][fila_img])
            n = int(ruta.stem.rsplit("_", 1)[1])          # archivos: <appid>_<n>.jpg, n = 0 es el header
            ax.imshow(Image.open(ruta))
            ax.set_title(f"{str(df.at[k, 'name'])[:35]} · {'header' if n == 0 else f'captura {n}'}",
                         fontsize=9)
            ax.axis("off")
        fig.suptitle("Par de imágenes más parecido según CLIP", fontsize=10)
        plt.show()
    print("Recordatorio: un parecido alto no demuestra copia. Es un candidato para revisión humana.")


if len(debiles[debiles.y == 1]):
    ejemplo = debiles[debiles.y == 1].iloc[0]
    explicar(int(ejemplo.i), int(ejemplo.j))

## 9. Buscar candidatos para un juego

El caso de uso de la persona usuaria: *"tengo mi juego, ¿qué juegos se le parecen tanto que vale la pena mirarlos?"*. Acepta el nombre o el AppID de Steam.

In [ ]:
def posicion(juego):
    if isinstance(juego, (int, np.integer)) or str(juego).isdigit():
        k = POS_DE_APPID.get(int(juego))
    else:
        k = POS_DE_NOMBRE.get(str(juego).strip().lower())
    if k is None:
        sug = df.loc[df["name"].str.lower().str.contains(str(juego).lower(), regex=False), "name"].head(8)
        raise KeyError(f"No encontré '{juego}'. Parecidos: {', '.join(sug) or 'ninguno'}")
    return k


def buscar(juego, k=10):
    q = posicion(juego)
    cand = sorted((set(IND_TFIDF[q]) | set(IND_EMB[q])) - {q})
    p = pd.DataFrame({"i": [min(q, c) for c in cand], "j": [max(q, c) for c in cand]})
    p = senal_imagen(senales_texto(p))
    p["otro"] = np.where(p["i"] == q, p["j"], p["i"])
    p["prob_A"] = modelo_A.predict_proba(p[FEAT_A])[:, 1]
    if modelo_B is not None:
        p["prob_B"] = modelo_B.predict_proba(p[FEAT_B])[:, 1]
    p["juego"] = df.loc[p["otro"], "name"].to_numpy()
    p["desarrollador"] = df.loc[p["otro"], "developer"].to_numpy()
    p["fecha"] = df.loc[p["otro"], "release_date"].dt.strftime("%Y-%m-%d").to_numpy()
    p["términos"] = [", ".join(terminos_compartidos(q, o, 5)) for o in p["otro"]]
    orden = "prob_B" if modelo_B is not None else "prob_A"
    p = p.assign(_r=p[orden] + p["tfidf_cos"]).sort_values("_r", ascending=False).head(k)
    cols = ["juego", "desarrollador", "fecha", "tfidf_cos", "emb_cos", "img_cos", "phash_sim", "titulo_sim",
            "prob_A"]
    cols += ["prob_B"] if modelo_B is not None else []
    print(f"Candidatos para: {df.at[q, 'name']} ({df.at[q, 'developer']})")
    return p[cols + ["términos"]].round(3).reset_index(drop=True)


buscar(df.at[int(debiles[debiles.y == 1].iloc[0].i), "name"] if len(debiles[debiles.y == 1]) else df.at[0, "name"])

## 10. Lote de pares para etiquetar a mano

El Modelo B y la evaluación que vale para el informe (sección 7) necesitan **pares etiquetados por personas**. Esta sección arma un lote de `N_LOTE` pares mezclando cuatro tipos, para que la evaluación no quede sesgada hacia lo que ya encuentra un método:

| Tipo | % | Por qué |
|---|---|---|
| `modelo` | 30 | Los que el Modelo A considera más probables entre los **sospechosos de reescritura** (significado parecido, palabras distintas). |
| `dudoso` | 20 | Probabilidad del Modelo A cerca de 0,5: son los que más le enseñan. |
| `texto` | 25 | Descripciones muy parecidas (`tfidf_cos ≥ 0.60`): lo que encontraría la línea base. |
| `azar` | 25 | Pares candidatos al azar: dan los negativos "normales". |

Se excluyen los pares del mismo desarrollador y los que ya tienen etiqueta (humana o débil). Las filas van **mezcladas y sin puntajes**, para que quien etiqueta no se deje llevar por el modelo; los puntajes quedan en otra hoja.

**Para el equipo:** sigan `docs/GUIA_ETIQUETADO.md`. En resumen: abran `salidas/lote_para_etiquetar.xlsx`, repártanse las filas, llenen `Etiqueta_humana` y `Responsable`, y guarden el archivo como `data/lote_etiquetado.xlsx`. La próxima corrida lo lee sola (sección 3).

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font
from openpyxl.worksheet.datavalidation import DataValidation

ETIQUETAS_LOTE = ["Coincidencia relevante", "Caso legítimo", "Descartado", "No concluyente"]
ya = set(zip(humanos["i"], humanos["j"])) | set(zip(debiles["i"], debiles["j"]))
libres = pares[(pares["mismo_dev"] != 1).to_numpy() & np.array([(i, j) not in ya for i, j in zip(pares["i"], pares["j"])])]
cuota = {t: round(N_LOTE * f) for t, f in [("modelo", .30), ("dudoso", .20), ("texto", .25), ("azar", .25)]}

# Sospechosos de reescritura con imágenes, para elegir por probabilidad del Modelo A
sosp = libres[(libres["emb_cos"] >= 0.80) & (libres["tfidf_cos"] < UMBRAL_DEBIL_POS)]
sosp = senal_imagen(sosp.sample(n=min(len(sosp), 8 * (cuota["modelo"] + cuota["dudoso"])), random_state=SEMILLA))
sosp["prob_A"] = modelo_A.predict_proba(sosp[FEAT_A])[:, 1] if len(sosp) else []
por_modelo = sosp.nlargest(cuota["modelo"], "prob_A").assign(tipo="modelo")
resto = sosp.drop(por_modelo.index)
dudosos = resto.iloc[(resto["prob_A"] - 0.5).abs().argsort()[:cuota["dudoso"]]].assign(tipo="dudoso")

usados = set(zip(por_modelo["i"], por_modelo["j"])) | set(zip(dudosos["i"], dudosos["j"]))
quedan = libres[[(i, j) not in usados for i, j in zip(libres["i"], libres["j"])]]
texto = quedan[quedan["tfidf_cos"] >= 0.60]
texto = texto.sample(n=min(len(texto), cuota["texto"]), random_state=SEMILLA).assign(tipo="texto")
quedan = quedan.drop(texto.index, errors="ignore")
# "azar" completa lo que falte si algún tipo no alcanzó su cuota
faltan = N_LOTE - len(por_modelo) - len(dudosos) - len(texto)
azar = quedan.sample(n=min(len(quedan), faltan), random_state=SEMILLA).assign(tipo="azar")

lote = senal_imagen(pd.concat([por_modelo, dudosos, texto, azar]).drop(columns=["img_cos", "phash_sim", "prob_A"],
                                                                       errors="ignore"))
lote["prob_A"] = modelo_A.predict_proba(lote[FEAT_A])[:, 1]
lote = lote.sample(frac=1, random_state=SEMILLA).reset_index(drop=True)   # mezclar los tipos
lote["ID_caso"] = [f"L-{n + 1:03d}" for n in range(len(lote))]


def url_steam(k):
    a = df.at[k, "appid"]
    return f"https://store.steampowered.com/app/{int(a)}" if pd.notna(a) else ""


def fecha(k):
    f = df.at[k, "release_date"]
    return f.date().isoformat() if pd.notna(f) else ""


hoja_pares = pd.DataFrame({
    "ID_caso": lote["ID_caso"],
    "juego_a": df.loc[lote["i"], "name"].to_numpy(), "url_a": [url_steam(k) for k in lote["i"]],
    "juego_b": df.loc[lote["j"], "name"].to_numpy(), "url_b": [url_steam(k) for k in lote["j"]],
    "dev_a": df.loc[lote["i"], "developer"].to_numpy(), "dev_b": df.loc[lote["j"], "developer"].to_numpy(),
    "fecha_a": [fecha(k) for k in lote["i"]], "fecha_b": [fecha(k) for k in lote["j"]],
    "Etiqueta_humana": "", "Responsable": "", "Observaciones": "",
    "appid_a": df.loc[lote["i"], "appid"].astype("Int64").to_numpy(),
    "appid_b": df.loc[lote["j"], "appid"].astype("Int64").to_numpy(),
})
hoja_senales = lote[["ID_caso", "tipo", "tfidf_cos", "emb_cos", "img_cos", "phash_sim", "titulo_sim",
                     "tags_jaccard", "prob_A"]].round(3)
instrucciones = pd.DataFrame({"Instrucciones": [
    "1. Abre los dos enlaces de Steam de cada fila y compara descripción, imágenes, mecánicas y desarrollador.",
    "2. Elige Etiqueta_humana en la lista: Coincidencia relevante (merece revisión: posible clon o reskin), "
    "Caso legítimo (se parecen, pero es normal: misma saga, mismo género, remaster, DLC), "
    "Descartado (no se parecen de verdad) o No concluyente (no se puede decidir; no se usa para entrenar).",
    "3. Escribe tu nombre en Responsable y, si quieres, el motivo en Observaciones.",
    "4. No mires la hoja 'Señales' antes de etiquetar: tiene los puntajes del modelo y te puede sesgar.",
    "5. Guarda el archivo como data/lote_etiquetado.xlsx en MyDrive/fairplay. Guía completa: docs/GUIA_ETIQUETADO.md",
]})

ruta_lote = SALIDA / "lote_para_etiquetar.xlsx"
with pd.ExcelWriter(ruta_lote, engine="openpyxl") as w:
    hoja_pares.to_excel(w, sheet_name="Pares", index=False)
    instrucciones.to_excel(w, sheet_name="Instrucciones", index=False)
    hoja_senales.to_excel(w, sheet_name="Señales", index=False)

# Lista desplegable en Etiqueta_humana, enlaces clicables y anchos legibles
libro = load_workbook(ruta_lote)
hp = libro["Pares"]
col = {c.value: c.column_letter for c in hp[1]}
lista = DataValidation(type="list", formula1='"' + ",".join(ETIQUETAS_LOTE) + '"', allow_blank=True)
hp.add_data_validation(lista)
lista.add(f"{col['Etiqueta_humana']}2:{col['Etiqueta_humana']}{len(hoja_pares) + 1}")
for nombre in ("url_a", "url_b"):
    for celda in hp[col[nombre]][1:]:
        if celda.value:
            celda.hyperlink, celda.font = celda.value, Font(color="0563C1", underline="single")
for letra, ancho in zip("ABCDEFGHIJKLMN", [9, 32, 14, 32, 14, 20, 20, 11, 11, 22, 14, 30, 9, 9]):
    hp.column_dimensions[letra].width = ancho
hp.freeze_panes = "B2"
libro["Instrucciones"].column_dimensions["A"].width = 140
libro.save(ruta_lote)

print(f"Lote de {len(lote)} pares en {ruta_lote}")
print(lote["tipo"].value_counts().to_string())
hoja_pares.head(10)

## 11. Exportar el paquete del modelo para la app

La app Gradio tiene una pestaña **"Modelo entrenado"** que usa este modelo sin volver a entrenar y **sin necesitar GPU ni CLIP**: todo lo pesado ya se calculó aquí. Esta celda deja en `salidas/paquete_app/` cuatro archivos:

| Archivo | Qué contiene |
|---|---|
| `modelo_pares.joblib` | Modelo A (y B si existe), la lista de señales que usa cada uno y la configuración. |
| `catalogo.parquet` | El catálogo ya limpio, **en el mismo orden** que los embeddings. |
| `emb_texto.npy` | Embeddings de texto de cada juego (en `float16` para que pese la mitad). |
| `imagenes.joblib` | Por `appid`: embeddings CLIP y hashes perceptuales de las imágenes ya procesadas. |

**Cómo usarlo:** copiar la carpeta `paquete_app` a `Primera fase práctica/modelo/` y abrir la app. En Hugging Face Spaces, subir esos cuatro archivos al dataset privado dentro de una carpeta `modelo/` (ver la guía técnica).

**Imágenes para la app.** El entrenamiento usa header + capturas solo de los juegos que necesita (unos 9 mil). Para que la app tenga señal de imagen en casi cualquier par, con `CABECERAS_APP = True` la primera celda de esta sección procesa **solo la imagen de cabecera** de todos los demás juegos (CLIP + hash, sin guardar los JPG). Tarda ~30 minutos la primera vez y queda en `salidas/cabeceras_v1.joblib`; si Colab se corta, la siguiente corrida sigue donde quedó. Esos juegos se comparan con una sola imagen en vez de cuatro, así que su `img_cos` tiende a ser algo menor.

Los juegos sin ninguna imagen funcionan igual: sus señales de imagen quedan vacías y el modelo las rellena (por eso el imputador agrega una columna "faltaba").

> **Importante:** las señales se calculan de la misma forma aquí y en `Primera fase práctica/senales.py`. Si cambian una, cambien la otra.

In [ ]:
# Imagen de cabecera de los juegos que no tienen imágenes procesadas (solo para la app)
cache_cab = SALIDA / "cabeceras_v1.joblib"
CAB = joblib.load(cache_cab) if cache_cab.exists() else {}   # appid -> {"emb": (1, 512) float16, "hash": (1, 64)}


def bajar_cabecera(k):
    a, url = df.at[k, "appid"], df.at[k, "header_image"]
    if not (isinstance(url, str) and url.startswith("http")):
        url = URL_HEADER.format(appid=int(a))
    try:
        r = requests.get(url, headers=CABECERAS, timeout=15)
        r.raise_for_status()
        return k, Image.open(io.BytesIO(r.content)).convert("RGB")
    except Exception:
        return k, None


if CABECERAS_APP:
    if _modelo_img is None:
        _modelo_img = SentenceTransformer(MODELO_IMAGEN, device=DISPOSITIVO)
    pendientes = [k for k in range(len(df)) if k not in IMGS and pd.notna(df.at[k, "appid"])
                  and int(df.at[k, "appid"]) not in CAB]
    print(f"Cabeceras en caché: {len(CAB):,} · por procesar: {len(pendientes):,}")
    t0, fallidas, LOTE_CAB = time.time(), 0, 1024
    with ThreadPoolExecutor(max_workers=16) as ex:
        for n, desde in enumerate(range(0, len(pendientes), LOTE_CAB)):
            bajadas = list(ex.map(bajar_cabecera, pendientes[desde:desde + LOTE_CAB]))
            ok = [(k, im) for k, im in bajadas if im is not None]
            fallidas += len(bajadas) - len(ok)
            if ok:
                V = _modelo_img.encode([im for _, im in ok], batch_size=128, normalize_embeddings=True,
                                       convert_to_numpy=True)
                for (k, im), v in zip(ok, V):
                    CAB[int(df.at[k, "appid"])] = {"emb": v[None, :].astype(np.float16),
                                                   "hash": hash_bits(im)[None, :]}
            if n % 20 == 19:   # guardar avance por si Colab se corta
                joblib.dump(CAB, cache_cab)
                hechas = desde + len(bajadas)
                print(f"  {hechas:,}/{len(pendientes):,} · {hechas / (time.time() - t0):.0f} por segundo")
    joblib.dump(CAB, cache_cab)
    print(f"Cabeceras listas: {len(CAB):,} ({fallidas:,} no se pudieron descargar) en {time.time() - t0:.0f} s")

PAQUETE = SALIDA / "paquete_app"
PAQUETE.mkdir(parents=True, exist_ok=True)

artefacto = {
    "modelo_A": modelo_A, "modelo_B": modelo_B, "FEAT_A": FEAT_A, "FEAT_B": FEAT_B,
    "modelo_texto": MODELO_TEXTO, "modelo_imagen": MODELO_IMAGEN,
    "config": {"UMBRAL_DEBIL_POS": UMBRAL_DEBIL_POS, "UMBRAL_DEBIL_EMB": UMBRAL_DEBIL_EMB,
               "MIN_TERMINOS": MIN_TERMINOS, "RANGO_DEBIL_NEG": RANGO_DEBIL_NEG,
               "K_VECINOS": K_VECINOS, "N_CAPTURAS": N_CAPTURAS, "n_juegos": len(df), "fuente": FUENTE},
    "n_debiles": len(debiles), "n_humanos": len(humanos),
    "sklearn": sklearn.__version__,  # la app revisa que coincida con la suya
}
joblib.dump(artefacto, PAQUETE / "modelo_pares.joblib")

columnas = ["appid", "name", "text", "developer", "release_date", "genres", "tags", "header_image", "n_resenas"]
df[columnas].to_parquet(PAQUETE / "catalogo.parquet", index=False)
np.save(PAQUETE / "emb_texto.npy", E.astype(np.float16))
img_app = {int(df.at[k, "appid"]): {"emb": v["emb"].astype(np.float16), "hash": v["hash"]}
           for k, v in IMGS.items() if pd.notna(df.at[k, "appid"])}
if CABECERAS_APP:   # las cabeceras solo completan juegos que no tenían imágenes
    ids_catalogo = set(df["appid"].dropna().astype(int))
    img_app |= {a: v for a, v in CAB.items() if a not in img_app and a in ids_catalogo}
joblib.dump(img_app, PAQUETE / "imagenes.joblib")

for f in sorted(PAQUETE.iterdir()):
    print(f"{f.name:22s} {f.stat().st_size / 1e6:8.1f} MB")
print(f"\nJuegos en el catálogo: {len(df):,} · con imágenes para la app: {len(img_app):,} "
      f"(header + capturas: {len(IMGS):,})")

## 12. Cómo contar estos resultados (informe y pitch)

* **Datos:** "Analizamos *N* juegos reales de Steam y *M* pares candidatos" (los números salen de las secciones 1 y 2).
* **Evidencia válida:** solo la tabla de la sección 7 (etiquetas humanas). La de la sección 6 muestra que el Modelo A aprende *algo* sin texto, pero sus etiquetas son automáticas.
* **Comparación honesta:** reporten la línea base TF-IDF al lado del modelo, con PR-AUC y precisión@20, e incluyan los casos donde el modelo se equivoca.
* **No digan** "detecta plagio", digan "prioriza pares para revisión humana".
* **Limitaciones a declarar:** las etiquetas débiles salen del texto, se usan pocas imágenes por juego (header y hasta `N_CAPTURAS` capturas), los datos de Steam no se publican en el repositorio, y la familia de un juego puede abarcar más de un grupo de validación.

**Siguientes pasos posibles:** comparar videos o más capturas y sumar Roblox o itch.io.